# LoopBuster Demo — Agent Loop Detection Walkthrough

**Goal:** See LoopBuster in action — detect when an LLM agent gets stuck in different kinds of loops, and understand how the detection works under the hood.

**Audience:** AI Agent engineers, LLM application developers, and anyone who's watched an agent burn through API credits by calling the same tool 50 times.

**Prerequisites:**
- Python 3.9+
- `pip install loopbuster`

**What you'll learn:**
- How to wrap any agent loop with LoopBuster
- How 4 detection strategies catch different loop patterns
- How noise denoising prevents false alarms from UUIDs and timestamps
- How adaptive thresholds handle agents with naturally repetitive behavior
- How to generate a Stuck Report for debugging
- How the Circuit Breaker works as a pre-flight gate

In [ ]:
# ════════════════════════════════════════════
# Setup: install and import
# ══════════════════════════════════════════════
!pip install loopbuster -q

from loopbuster import LoopBuster, AsyncLoopBuster
from loopbuster.types import Action, ActionConfig, AdaptiveActionConfig
from loopbuster.strategies import (
    ExactRepeatStrategy, FuzzyRepeatStrategy,
    CycleDetectionStrategy, OutputStagnationStrategy,
)
from loopbuster.types import ActionRecord
from loopbuster.circuit import CircuitBreaker, BreakerAction
from loopbuster.similarity import args_similarity

print('LoopBuster ready \U0001f6d1')

---
## 1. Quick Start: Catch an Exact Repeat

The simplest loop: an agent calls the same tool with the same arguments over and over. LoopBuster's `ExactRepeatStrategy` catches this immediately.

In [ ]:
# Simulate an agent stuck calling web_search with the same query
buster = LoopBuster()

print("Simulating a stuck agent...\n")

for i in range(8):
    decision = buster.check(tool="web_search", args={"q": "python error"})
    status = (
        "\U0001f534 LOOP" if decision.is_loop else
        "\U0001f7e1 WARN" if decision.should_warn else
        "\U0001f7e2 OK"
    )
    print(f"  Step {i+1}: {status}  |  {decision.reason[:60] if decision.reason else 'first call, learning...'}")
    if decision.is_loop:
        print(f"\n  \U0001f6d1 Stopped after {i+1} calls. Confidence: {decision.confidence:.0%}")
        break

**What happened?**

- Step 1-2: learning phase — no history yet
- Step 3: `ExactRepeat` detects 3 consecutive identical calls (confidence 0.6), engine WARNS
- Step 4: engine STOPS — agent is clearly stuck

The engine escalates from ALLOW → WARN → STOP → ESCALATE as `consecutive_hits` accumulate.

---
## 2. Fuzzy Repeat: When Arguments Change Slightly

Real agents don't always repeat *exact* arguments — they modify the query slightly each time. `FuzzyRepeatStrategy` catches this using **multi-factor similarity**.

**The similarity engine:**
- Jaccard similarity on tokens
- Normalized Levenshtein edit distance
- Dict structure similarity (key overlap + type matching)
- **Noise denoising**: UUIDs, timestamps, hashes, and volatile fields (IDs, tokens) are normalized before comparison

In [ ]:
# Demo: similarity in action
print("Similarity scores between args pairs:\n")

# Two queries with different volatile fields (should be 1.0 after denoising)
a1 = {"q": "python error", "request_id": "550e8400-e29b-41d4-a716-446655440000"}
a2 = {"q": "python error", "request_id": "6ba7b810-9dad-11d1-80b4-00c04fd430c8"}
sim = args_similarity(a1, a2)
print(f"  UUID-denoised args: {sim:.2f}  (identical after denoising)")

# Two genuinely different queries
a3 = {"q": "python error fix"}
a4 = {"q": "machine learning tutorial"}
sim = args_similarity(a3, a4)
print(f"  Different queries:    {sim:.2f}  (correctly low)")

# Same query, different pagination (structurally similar)
a5 = {"q": "weather Tokyo", "page": 1}
a6 = {"q": "weather Tokyo", "page": 2}
sim = args_similarity(a5, a6)
print(f"  Pagination variant:   {sim:.2f}  (high structure similarity)")

In [ ]:
# Simulate an agent that keeps searching with slightly different request IDs
buster = LoopBuster(similarity_threshold=0.75)

print("Agent stuck searching with different request IDs...\n")

for i in range(5):
    # Each call has a unique UUID in the request_id field
    import uuid
    args = {"q": "python error", "request_id": str(uuid.uuid4())}
    decision = buster.check(tool="web_search", args=args)
    status = "\U0001f534 LOOP" if decision.is_loop else "\U0001f7e1 WARN" if decision.should_warn else "\U0001f7e2 OK"
    strategy = f"[{decision.strategy}]" if decision.strategy else ""
    print(f"  Step {i+1}: {status} {strategy}  |  {decision.reason[:55] if decision.reason else 'learning...'}")
    if decision.is_loop:
        print(f"\n  \U0001f6d1 Detected! UUID differences were stripped by denoising.")
        break

**Key insight:** The request IDs are all different — a naive string match would miss this loop entirely. LoopBuster's denoising pipeline strips UUIDs (`550e8400-...` → `<UUID>`) and volatile keys (`request_id` → `<VOLATILE>`), making the args *identical* after normalization.

---
## 3. Cycle Detection: A→B→A→B Pattern

Some agents don't repeat the same tool — they cycle through a sequence: search → parse → search → parse → ... `CycleDetectionStrategy` catches repeating tool sequences.

In [ ]:
buster = LoopBuster(action_config=ActionConfig(warn_threshold=1, stop_threshold=2))

print("Agent cycling through search→parse→search→parse...\n")

tools = ["web_search", "parse_results", "web_search", "parse_results",
         "web_search", "parse_results", "web_search", "parse_results"]

for i, tool in enumerate(tools):
    decision = buster.check(tool=tool, args={"q": "python"})
    status = "\U0001f534 LOOP" if decision.is_loop else "\U0001f7e1 WARN" if decision.should_warn else "\U0001f7e2 OK"
    print(f"  Step {i+1}: {status} [{tool:15s}]  {decision.reason[:50] if decision.reason else ''}")
    if decision.is_loop:
        print(f"\n  \U0001f6d1 Cycle detected after {i+1} steps!")
        break

---
## 4. Output Stagnation: Same Result, Different Tools

An agent might call different tools or different queries but get the same result — this means it's stuck even though the calls look different.

In [ ]:
buster = LoopBuster()

print("Agent getting the same result from every call...\n")

queries = ["python error", "python bug", "python fix", "python issue"]
for i, q in enumerate(queries):
    decision = buster.check(tool="web_search", args={"q": q}, output="Error: timeout")
    status = "\U0001f534 LOOP" if decision.is_loop else "\U0001f7e1 WARN" if decision.should_warn else "\U0001f7e2 OK"
    print(f"  Step {i+1}: {status} q='{q}'  |  {decision.reason[:50] if decision.reason else ''}")
    if decision.is_loop:
        print(f"\n  \U0001f6d1 Stagnation detected! Every query returned 'Error: timeout'.")
        break

---
## 5. Adaptive Thresholds: Smart Escalation

A fixed threshold that works for a search agent will false-alarm on a coding agent that naturally calls the same tools repeatedly. `AdaptiveActionConfig` adjusts thresholds based on **action diversity**:

- **High diversity** (many different tools) → thresholds relax (fewer false positives)
- **Low diversity** (repeating the same tools) → thresholds tighten (faster intervention)

In [ ]:
from loopbuster.types import AdaptiveActionConfig

cfg = AdaptiveActionConfig(base_warn=3, base_stop=5, base_escalate=8, diversity_window=10)

print("=== High Diversity Scenario ===")
print("Agent doing 10 different tasks (diverse actions):")
for t in ["search", "read", "write", "api", "search", "parse", "transform", "load", "query", "save"]:
    cfg.record_action(t)
print(f"  Diversity ratio: {cfg.diversity_ratio:.2f}")
print(f"  Thresholds: WARN>={cfg.warn_threshold}, STOP>={cfg.stop_threshold}, ESC>={cfg.escalate_threshold}")
print(f"  (Relaxed: needs more repetitions before intervening)")

print()
cfg.reset()

print("=== Low Diversity Scenario ===")
print("Agent doing the same thing 10 times (stuck):")
for _ in range(10):
    cfg.record_action("web_search")
print(f"  Diversity ratio: {cfg.diversity_ratio:.2f}")
print(f"  Thresholds: WARN>={cfg.warn_threshold}, STOP>={cfg.stop_threshold}, ESC>={cfg.escalate_threshold}")
print(f"  (Tightened: intervenes much sooner)")

**Why this matters:** In production, agents have legitimate repetitive patterns (e.g., a coding agent calling `read_file` 20 times). Adaptive thresholds prevent false alarms while still catching real loops.

---
## 6. Stuck Report: Post-Mortem Diagnostics

After an agent run, you can generate a diagnostic report showing diversity ratio, top repeated patterns, token waste estimates, and recommendations.

In [ ]:
buster = LoopBuster(budget_usd=5.0)

# Simulate a stuck agent with some token usage
for i in range(15):
    buster.check(tool="web_search", args={"q": "python error"})
    buster.record_tokens("gpt-4o-mini", input=500, output=200)

# Generate the report
report = buster.report()

print("\U0001f4cb LoopBuster Stuck Report")
print("=" * 40)
print(f"Total actions:          {report['total_actions']}")
print(f"Unique signatures:      {report['unique_signatures']}")
print(f"Diversity ratio:        {report['diversity_ratio']:.2f}")
print(f"Redundant actions:      {report['redundant_actions']}")
print(f"Estimated token waste:  {report['estimated_token_waste']:,} tokens")
print(f"Estimated cost waste:   \${report['estimated_cost_waste_usd']:.6f}")
print(f"Consecutive hits:       {report['consecutive_hits']}")
print(f"Spent USD:              \${report['spent_usd']:.4f}")
print()
print("Top repeated patterns:")
for sig, cnt in report['top_repeated_patterns'][:3]:
    print(f"  {sig}: {cnt}x")
print()
print("Recommendations:")
for rec in report['recommendations'][:3]:
    print(f"  \U0001f4a1 {rec}")

---
## 7. Circuit Breaker: Pre-Flight Check

The Circuit Breaker is a **pre-flight gate**: you can check *before* calling a tool whether that call would be blocked. It's an exact-match gate (not fuzzy), useful for hard limits.

In [ ]:
from loopbuster.circuit import CircuitBreaker, BreakerAction

breaker = CircuitBreaker(max_repeats=3, action=BreakerAction.BLOCK)
buster = LoopBuster(circuit_breaker=breaker)

print("Pre-flight check demo (max_repeats=3, action=BLOCK):\n")

for i in range(6):
    # Pre-flight check before calling
    pre = buster.breaker_check("web_search", {"q": "python"})
    
    if pre.blocked:
        print(f"  Attempt {i+1}: \U0001f6ab BLOCKED | {pre.reason}")
        print(f"             Alternative: {pre.alternative_suggestion}")
        break
    else:
        # Proceed with the call and record it
        decision = buster.check(tool="web_search", args={"q": "python"})
        what = "\U0001f7e2 OK" if decision.action == Action.ALLOW else "\U0001f7e1 WARN"
        print(f"  Attempt {i+1}: {what} | {pre.reason}")

---
## 8. Putting It All Together: Realistic Agent Loop

Let's simulate a realistic scenario: a medical diagnosis agent processing oral images. It gets stuck because the vision model keeps returning low confidence, and the agent keeps retrying.

In [ ]:
import time
from dataclasses import dataclass

@dataclass
class Step:
    tool: str
    args: dict
    output: str

# Simulate an agent stuck on image analysis
def stuck_medical_agent():
    """A generator that simulates a stuck agent."""
    image_ids = ["IMG-001", "IMG-002", "IMG-003", "IMG-004"]
    while True:
        for img_id in image_ids:
            yield Step(
                tool="analyze_oral_image",
                args={"image_id": img_id, "analysis_type": "caries_detection"},
                output="Confidence: 0.45 — below threshold, retrying..."
            )

print("Medical diagnosis agent getting stuck...\n")

buster = LoopBuster(
    similarity_threshold=0.75,
    action_config=ActionConfig(warn_threshold=1, stop_threshold=2, escalate_threshold=3),
)

for step_num, step in enumerate(stuck_medical_agent()):
    decision = buster.check(tool=step.tool, args=step.args, output=step.output)
    
    icons = {Action.ALLOW: "\U0001f7e2", Action.WARN: "\U0001f7e1", 
             Action.STOP: "\U0001f534", Action.ESCALATE: "\U0001f6a8"}
    icon = icons.get(decision.action, "\u2753")
    
    print(f"  {icon} Step {step_num+1}: {step.tool}({step.args['image_id']})")
    
    if decision.should_warn:
        print(f"     \u26a0\ufe0f  {decision.reason}")
    if decision.is_loop:
        print(f"     \U0001f6d1  {decision.reason}")
        print(f"\n  Intervened after {step_num+1} steps. Action: {decision.action.name}")
        break
    
    if step_num > 20:
        print("  (safety break)")
        break

**What happened?**

The agent cycles through 4 image IDs (IMG-001 → IMG-004 → IMG-001 → ...). `CycleDetectionStrategy` recognizes the 4-step repeating pattern and escalates. The engine then recommends switching to a different analysis approach.

In production, you'd use the `on_stop` callback to trigger a recovery action — e.g., switch models, ask for human input, or try a fallback tool.

---
## What You've Learned

| Concept | Demo |
|---|---|
| Exact repeat detection | Agent calling web_search with the same query → caught by ExactRepeatStrategy |
| Noise denoising | UUIDs and timestamps stripped before comparison → args become identical |
| Fuzzy similarity | Jaccard + Levenshtein + dict structure → catches near-identical args |
| Cycle detection | A→B→A→B tool sequence → recognized as repeating pattern |
| Output stagnation | Different queries, same result → caught by OutputStagnationStrategy |
| Adaptive thresholds | Diversity-aware tightening/relaxing → fewer false positives |
| Stuck report | Post-mortem diagnostics: diversity, waste estimate, recommendations |
| Circuit breaker | Pre-flight check before tool calls → BLOCK/WARN/SUGGEST |

---

*Built with [LoopBuster](https://github.com/liuchunwei732-cmyk/loopbuster) — Break the infinite loops of your AI Agents.*